In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin1")
print("Dataset shape:", df.shape)
df.head()
df["fraud"] = np.where(df["Order Status"] == "SUSPECTED_FRAUD", 1, 0)
df["late_delivery"] = np.where(df["Delivery Status"] == "Late delivery", 1, 0)

df[["Order Status", "Delivery Status", "fraud", "late_delivery"]].head()
categorical_cols = df.select_dtypes(include="object").columns

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))
df = df.fillna(df.median(numeric_only=True))
remove_cols = ["fraud", "late_delivery", "Customer Zipcode",
               "Order Id", "Customer Id", "Product Description"]

X = df.drop(remove_cols, axis=1)
y_fraud = df["fraud"]
y_late = df["late_delivery"]
splits = [(0.2, "80/20"), (0.3, "70/30")]
models = {
    "Logistic Regression": LogisticRegression(max_iter=300),
    "Random Forest": RandomForestClassifier(
        n_estimators=50,      # rapide
        max_depth=10,         # rapide
        n_jobs=-1             # utilise tous les CPU
    )
}

def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return (
        accuracy_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    )
results_fraud = {}

for test_size, split_name in splits:
    X_train, X_test, y_train_f, y_test_f = train_test_split(
        X, y_fraud, test_size=test_size, random_state=42
    )

    temp = {}

    for name, model in models.items():
        acc, rec, f1 = evaluate(model, X_train, X_test, y_train_f, y_test_f)
        temp[name] = [acc, rec, f1]

    results_fraud[split_name] = temp

results_fraud
results_late = {}

for test_size, split_name in splits:
    X_train, X_test, y_train_l, y_test_l = train_test_split(
        X, y_late, test_size=test_size, random_state=42
    )

    temp = {}

    for name, model in models.items():
        acc, rec, f1 = evaluate(model, X_train, X_test, y_train_l, y_test_l)
        temp[name] = [acc, rec, f1]

    results_late[split_name] = temp

results_late
best_split = max(results_fraud, key=lambda s:
                 np.mean([results_fraud[s][m][2] for m in models]))
best_split
final_table = []

for model in models:
    acc_f, rec_f, f1_f = results_fraud[best_split][model]
    acc_l, rec_l, f1_l = results_late[best_split][model]

    final_table.append([
        model, acc_f, rec_f, f1_f, acc_l, rec_l, f1_l
    ])

columns = ["Model", "Acc (Fraud)", "Recall (Fraud)", "F1 (Fraud)",
           "Acc (Late)", "Recall (Late)", "F1 (Late)"]

final_df = pd.DataFrame(final_table, columns=columns)
final_df
rf = RandomForestClassifier(n_estimators=50, max_depth=10)
rf.fit(X, y_fraud)

importances = pd.Series(rf.feature_importances_, index=X.columns)
top5_fraud = importances.sort_values(ascending=False).head(5)

plt.figure(figsize=(6,4))
top5_fraud.plot(kind="bar")
plt.title("Top 5 Features — Fraud Detection")
plt.show()
rf2 = RandomForestClassifier(n_estimators=50, max_depth=10)
rf2.fit(X, y_late)

importances2 = pd.Series(rf2.feature_importances_, index=X.columns)
top5_late = importances2.sort_values(ascending=False).head(5)

plt.figure(figsize=(6,4))
top5_late.plot(kind="bar")
plt.title("Top 5 Features — Late Delivery")
plt.show()



Dataset shape: (180519, 53)


KeyboardInterrupt: 

In [36]:
final_df


,Model,Acc (Fraud),Recall (Fraud),F1 (Fraud),Acc (Late),Recall (Late),F1 (Late)
0,Logistic Regression,0.976457,0.0,0.0,0.548333,1.0,0.708288
1,Random Forest,1.000000,1.0,1.0,1.000000,1.0,1.000000
